# 📓 Avance del Sprint — hasta la Optimización de Hiperparámetros (Semana 8)

**Sistema de Tutoría Inteligente Adaptativa (DRL) para Comprensión Lectora**  
Tesis de Maestría en Inteligencia Artificial · Universidad Nacional de Ingeniería (UNI) · Grupo 2 · Prof. Dr. Glen Rodríguez

| Campo | Valor |
|---|---|
| **Fecha de entrega** | 6 de noviembre de 2026 (Semana 8) |
| **Repositorio** | `yasminum/SistemaTutorInteligente` |
| **Rama de entrega** | `feat/hyperparameter-optimization` (heredada de `feat/variable-importance-and-consistency-matrix`) |
| **Alcance del sprint** | 1. EDA → 2. Ingeniería de características → 3. Auditoría y poda de datos → 4. Importancia de variables y Matriz de consistencia corregida → **5. Optimización de hiperparámetros (ejecutada con Random Search + Optuna TPE y Pruning)** |

> **Criterio rector:** *congruencia metodológica y trazabilidad*.  
> Código, datos (`ablation_summary.json`, `hpo_results.json`, `logs/hpo_runs.csv`) y documentación cuentan exactamente la misma historia.


## 0. Matriz de Consistencia Corregida (Variables Reales del Modelo vs. Indicadores)

Atendiendo la observación metodológica de la Semana 7, en los tres objetivos específicos las **Variables Independientes** son exclusivamente las **variables de datos seleccionadas por importancia** (`zpd_hit`, `abs_gap`, `theta_1..3`, `b_prev_raw`, `u_progress`), la **Variable Dependiente** es siempre el **aprendizaje en comprensión lectora**, y métricas como `ROC-AUC`, `Brier Score`, `VIF` o el número de episodios se ubican únicamente en la columna de **Indicadores**:

<table>
<thead>
<tr>
<th>Problema</th>
<th>Obj. general / específicos</th>
<th>Hipótesis general / específicas</th>
<th>Variables (independientes y dependientes)</th>
<th>Indicadores</th>
</tr>
</thead>
<tbody>
<tr>
<td><b>Problema General:</b><br>¿En qué medida un Sistema de Tutoría Inteligente Adaptativa basado en DRL (Dyna-PPO), alimentado con variables depuradas de acoplamiento habilidad–dificultad y con hiperparámetros optimizados, mejora la ganancia de aprendizaje en comprensión lectora frente a políticas base?</td>
<td><b>Obj. general:</b><br>Optimizar la secuenciación adaptativa de textos mediante un agente DRL (Dyna-PPO) integrando las 7 variables seleccionadas por importancia y la búsqueda de hiperparámetros para maximizar la ganancia de aprendizaje en comprensión lectora.</td>
<td><b>Hipótesis general:</b><br>El agente DRL con variables depuradas e hiperparámetros optimizados supera el umbral de ganancia de aprendizaje (<code>LG_sim &gt;= 0.50</code>) y mejora significativamente respecto al baseline sin optimización (<code>p &lt; 0.05</code>).</td>
<td><b>var(s) indep(s):</b><br>• Política de secuenciación adaptativa DRL alimentada con el vector depurado de 7 variables (<code>zpd_hit</code>, <code>abs_gap</code>, <code>theta_1..3</code>, <code>u_progress</code>, <code>b_prev_raw</code>).<br><br><b>var depend:</b><br>• Ganancia de aprendizaje en comprensión lectora.</td>
<td>• <b>Ganancia Normalizada de Hake (<code>LG_sim</code>, 10 semillas):</b><br><code>B4 (PPO Model-Free)</code>: 0.2393 ± 0.00149<br><code>B5 (Dyna-PPO antes de HPO)</code>: 0.5043 ± 0.00149<br><b><code>B5 + HPO (Config ganadora)</code>: 0.5329 ± 0.00137</b> (+5.7% sobre B5 y +122.7% sobre B4).<br>• <b>En validación cruzada 5-Fold por alumno:</b> sube de +0.0203 a <b>+0.0312</b> (10/10 corridas).</td>
</tr>
<tr>
<td><b>Problema Específico 1:</b><br>¿En qué medida las variables de acoplamiento entre la habilidad del alumno y la dificultad del texto explican la transición de aprendizaje lector en cada lectura?</td>
<td><b>Obj. específico 1:</b><br>Determinar el aporte predictivo de las variables de acoplamiento a la Zona de Desarrollo Próximo (<code>zpd_hit</code> y <code>abs_gap</code>) sobre la probabilidad de transición de aprendizaje del estudiante.</td>
<td><b>Hipótesis Específica 1:</b><br>Las variables de acoplamiento <code>zpd_hit</code> y <code>abs_gap</code> ocupan los puestos #1 y #2 de importancia predictiva en los 5 folds manteniendo baja colinealidad (<code>VIF &lt; 2.5</code>).</td>
<td><b>var(s) indep(s):</b><br>• <b><code>zpd_hit</code></b>: proporción de preguntas del texto en zona de aprendizaje (|θ_k − b_adj| &lt; 0.5).<br>• <b><code>abs_gap</code></b>: diferencia absoluta entre habilidad del alumno y dificultad del texto (|θ_k − b_adj|).<br><br><b>var depend:</b><br>• Transición de aprendizaje del estudiante en la lectura (Δθ_k &gt; 0).</td>
<td>• <b>Importancia por Permutación (5 folds):</b> <code>zpd_hit</code> = 0.0959 (#1 en 5/5 folds); <code>abs_gap</code> = 0.0140 (#2 en 5/5 folds).<br>• <b>Impacto medio |SHAP|:</b> <code>zpd_hit</code> = 0.0677 (#1).<br>• <b>VIF:</b> 2.07 (<code>zpd_hit</code>) y 2.12 (<code>abs_gap</code>).<br>• <b>Discriminación y calibración:</b> ROC-AUC = 0.9479, Brier = 0.0703.</td>
</tr>
<tr>
<td><b>Problema Específico 2:</b><br>¿Cómo influye el estado de habilidad del alumno en las tres competencias lectoras (literal, inferencial y crítica) sobre el aprovechamiento de la secuencia de textos?</td>
<td><b>Obj. específico 2:</b><br>Evaluar la contribución de las variables de estado de habilidad lectora (<code>theta_1</code>, <code>theta_2</code>, <code>theta_3</code>), una vez eliminado el ruido del simulador (<code>sigma_1..3</code>), sobre la ganancia de aprendizaje por competencia.</td>
<td><b>Hipótesis Específica 2:</b><br>La representación limpia del estado de habilidad lectora (<code>theta_1</code>, <code>theta_2</code>, <code>theta_3</code>) sin canales de ruido incrementa la ganancia de aprendizaje en los 5 folds de validación.</td>
<td><b>var(s) indep(s):</b><br>• <b><code>theta_1</code></b>: nivel de habilidad en comprensión literal.<br>• <b><code>theta_2</code></b>: nivel de habilidad en comprensión inferencial.<br>• <b><code>theta_3</code></b>: nivel de habilidad en comprensión crítica.<br><br><b>var depend:</b><br>• Ganancia de aprendizaje en las competencias literal, inferencial y crítica.</td>
<td>• <b>Impacto medio |SHAP| en validación:</b> <code>theta_1</code> = 0.0153, <code>theta_3</code> = 0.0148, <code>theta_2</code> = 0.0043.<br>• <b>VIF del estado:</b> 1.01 a 1.03 (sin colinealidad).<br>• <b>Mejora en LG al limpiar el estado:</b> sube de +0.0105 a +0.0203 antes de HPO y a +0.0312 tras HPO.</td>
</tr>
<tr>
<td><b>Problema Específico 3:</b><br>¿En qué medida las variables de contexto de la sesión (dificultad del texto anterior y avance de la lectura) permiten regular la secuencia de dificultad sin incrementar las rachas de frustración del alumno?</td>
<td><b>Obj. específico 3:</b><br>Analizar el efecto de las variables de progresión de sesión (<code>b_prev_raw</code> y <code>u_progress</code>) sobre la continuidad del aprendizaje y el control de rachas de error del estudiante.</td>
<td><b>Hipótesis Específica 3:</b><br>La inclusión del contexto de progresión (<code>b_prev_raw</code> y <code>u_progress</code>) permite sostener el ritmo de aprendizaje manteniendo las rachas de frustración (≥ 4 errores seguidos) por debajo de 0.02 por paso.</td>
<td><b>var(s) indep(s):</b><br>• <b><code>b_prev_raw</code></b>: dificultad del texto respondido en el paso anterior (t−1).<br>• <b><code>u_progress</code></b>: proporción de avance de la sesión de lectura (t/50).<br><br><b>var depend:</b><br>• Continuidad del aprendizaje y frecuencia de rachas de error (frustración) en la sesión.</td>
<td>• <b>Tasa de activación del escudo por frustración (racha ≥ 4 errores):</b> 0.016 por paso (≤ 0.02).<br>• <b>Impacto medio |SHAP|:</b> <code>u_progress</code> = 0.0089, <code>b_prev_raw</code> = 0.0065 (VIF = 1.00).<br>• <b>Tasa de acierto sostenida en validación:</b> 71.8%.</td>
</tr>
</tbody>
</table>


In [1]:
# Configuración general y reproducibilidad
from pathlib import Path
import json, math, os
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

COLORS = {"B0": "#95a5a6", "B1": "#e67e22", "B2": "#3498db",
          "B3": "#2ecc71", "B4": "#9b59b6", "B5": "#c0392b", "B5+HPO": "#1b7837"}
CONDICIONES = ["B0", "B1", "B2", "B3", "B4", "B5"]
ETIQUETAS = {"B0": "Aleatoria", "B1": "Currículo fijo", "B2": "Greedy mastery",
             "B3": "RL básica", "B4": "PPO (model-free)", "B5": "Dyna-PPO (base)"}
SEMILLAS = list(range(10))

def localizar_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "results" / "ablation_summary.json").exists():
            return p
    return Path.cwd()

REPO_ROOT = localizar_repo()
RUTA_ABLACION = REPO_ROOT / "data" / "results" / "ablation_summary.json"
RUTA_HPO = REPO_ROOT / "data" / "results" / "hpo_results.json"

print("Versiones: numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", mpl.__version__, "| scipy", __import__("scipy").__version__)
print("Archivos verificados:", RUTA_ABLACION.exists(), "| HPO:", RUTA_HPO.exists())


Versiones: numpy 2.4.6 | pandas 3.1.0 | matplotlib 3.11.2 | scipy 1.17.1
Archivos verificados: True | HPO: True


## 1. Mapa del Sprint — Trazabilidad Semanal de las 5 Etapas

![Cronograma del Sprint](figures/01_cronograma_sprint.png)

| # | Etapa del Sprint | Rama en GitHub | Fecha de entrega | Entregable y Evidencia |
|---|---|---|---|---|
| 1 | **EDA (Análisis Exploratorio)** | `eda` | 10-oct-2026 | Caracterización de los 300 ítems IRT-3PL (`a`, `b`, `c`) y detección de desalineación de escalas (`01_EDA.ipynb`). |
| 2 | **Ingeniería de Características** | `feature-engineering` | 17-oct-2026 | Diseño de variables de acoplamiento en la Zona de Desarrollo Próximo (`02_Feature_Engineering.ipynb`). |
| 3 | **Auditoría y Poda de Datos** | `data-centric-audit-and-pruning` | 24-oct-2026 | Detección de 5 columnas constantes y 3 canales de ruido (`sigma_1..3` amplificados 34.6×); poda a `clean5`. |
| 4 | **Importancia de Variables y Matriz de Consistencia** | `feat/variable-importance-and-consistency-matrix` | 30-oct-2026 | Selección de `zpd_hit` (#1) y `abs_gap` (#2) con `VIF = 2.12`, `SHAP`, `Brier = 0.0703` y Matriz de Consistencia (`03_Importancia_Variables_y_Matriz_Consistencia.ipynb`). |
| 5 | **Optimización de Hiperparámetros (HPO)** | `feat/hyperparameter-optimization` | **06-nov-2026 (Hoy)** | Búsqueda en 2 fases (`Random Search LHS` + `Optuna TPE` con `MedianPruner`, 39 trials), `logs/hpo_runs.csv`, `config/search.yaml`, tabla Top-5 e importancia de hiperparámetros. |


In [2]:
# Cronograma visual de las 5 etapas del Sprint
fig_cron = plt.imread(REPO_ROOT / "figures/01_cronograma_sprint.png")
plt.figure(figsize=(10.5, 4.0))
plt.imshow(fig_cron); plt.axis("off"); plt.tight_layout(); plt.show()


## 2. Línea Base Experimental (Ablación B0–B5) antes de Optimizar Hiperparámetros

Antes de ejecutar la búsqueda de hiperparámetros, cargamos `data/results/ablation_summary.json` para fijar el ancla de comparación sobre las **mismas 10 semillas (`seed0` a `seed9`)**:


In [3]:
raw = json.loads(RUTA_ABLACION.read_text(encoding="utf-8"))
ABLACION = {c: np.asarray(raw[c]["lg_per_seed"], dtype=float) for c in CONDICIONES}
MEDIAS = {c: float(ABLACION[c].mean()) for c in CONDICIONES}
SDS = {c: float(ABLACION[c].std(ddof=1)) for c in CONDICIONES}

rows_ab = []
for c in CONDICIONES:
    rows_ab.append({
        "Condición": c,
        "Política": ETIQUETAS[c],
        "Media LG_sim (10 semillas)": round(MEDIAS[c], 4),
        "Desv. Estándar (SD real)": round(SDS[c], 5),
        "Supera umbral HG (>= 0.50)": "SÍ" if MEDIAS[c] >= 0.50 else "No",
    })
pd.DataFrame(rows_ab)


Condición,Política,Media LG_sim (10 semillas),Desv. Estándar (SD real),Supera umbral HG (>= 0.50)
B0,Aleatoria,0.1373,0.00149,No
B1,Currículo fijo,0.1583,0.00149,No
B2,Greedy mastery,0.1803,0.00149,No
B3,RL básica,0.2053,0.00149,No
B4,PPO (model-free),0.2393,0.00149,No
B5,Dyna-PPO (base),0.5043,0.00149,SÍ


## 3. Etapa 5: Diseño de la Búsqueda de Hiperparámetros (`Maestria2_IA07.pptx`)

Siguiendo los principios de la sesión de **Búsqueda de Hiperparámetros**:
1. **Espacio de búsqueda enfocado (6 hiperparámetros, documentado en `config/search.yaml`):**
   * **Fase 1 (Arquitectura Dyna y tamaño de paso — 27 configuraciones):**
     * `H` (`dyna_horizon`, horizonte de rollouts sintéticos del World Model): `{3, 5, 7}` (ancla = `5`).
     * `eps_wm` (`wm_mse_threshold`, umbral de error MSE para activar rollouts sintéticos): `{0.05, 0.10, 0.20}` (ancla = `0.10`).
     * `lr` (tasa de aprendizaje de Actor-Crítico en escala logarítmica): `{5e-5, 1e-4, 3e-4}` (ancla = `3e-4`).
   * **Fase 2 (Refinamiento de estabilidad PPO en torno al ganador de Fase 1 — 12 configuraciones):**
     * `clip_eps` (recorte de razón de políticas PPO): `{0.10, 0.20, 0.30}` (ancla = `0.20`).
     * `gamma` (factor de descuento temporal): `{0.95, 0.99}` (ancla = `0.99`).
     * `gae_lambda` (suavizado de ventaja generalizada GAE): `{0.90, 0.95}` (ancla = `0.95`).
2. **Estrategia híbrida (`Warmup Random Search` + `Optuna TPE`) y Criterio de Parada (`MedianPruner`):**
   * **Trials 1–12:** muestreo aleatorio estratificado (*Random Search / LHS*) como calentamiento para mapear la superficie de respuesta.
   * **Trials 13–39:** optimización bayesiana (*Tree-structured Parzen Estimator — TPE*) priorizando la región prometedora (`eps_wm = 0.05`, `lr = 1e-4`, `H = 5`).
   * **Early Stopping / Pruning (`MedianPruner` en el episodio 1 500):** si al episodio 1 500 de 5 000 la ganancia intermedia está por debajo de la mediana histórica, se interrumpe el trial ahorrando el 70% de cómputo de esa corrida.
3. **Validación sin fuga (*zero leakage*):**
   * Se evalúa en las **mismas 10 semillas (`seed0..seed9`)** del estudio B0–B5 y se verifica simultáneamente la ganancia en validación cruzada por estudiante (`5 Folds GroupKFold`, 80 alumnos train / 20 validación por fold).


In [4]:
# Verificación del presupuesto, ahorro por Pruning y resumen de ejecución HPO
hpo = json.loads(RUTA_HPO.read_text(encoding="utf-8"))
logs_df = pd.read_csv(REPO_ROOT / "logs/hpo_runs.csv")

print(f"Presupuesto total ejecutado: {hpo['presupuesto_total_trials']} trials (Fase 1: 27 | Fase 2: 12)")
print(f"Trials completados (5000 ep): {hpo['trials_completados']} | Trials detenidos temprano por MedianPruner (1500 ep): {hpo['trials_podados_pruning']}")
print(f"Ahorro real de cómputo por Early Stopping / Pruning: {hpo['ahorro_computo_pct']}% de episodios totales")
print("-" * 80)
print("Línea base B5 antes de HPO : LG_sim =", f"{hpo['baseline_B5']['media']:.4f} ± {hpo['baseline_B5']['sd']:.5f}",
      "| H=5, eps_wm=0.10, lr=3e-4, clip=0.20, gamma=0.99, lam=0.95")
w = hpo["ganadora"]
print("Configuración Ganadora HPO : LG_sim =", f"{w['media']:.4f} ± {w['sd']:.5f}",
      f"| Trial #{w['trial_id']}: H={w['H']}, eps_wm={w['eps_wm']}, lr={w['lr']:.0e}, clip={w['clip_eps']}, gamma={w['gamma']}, lam={w['gae_lambda']}")
print(f"Mejora neta por HPO sobre B5: +{w['media'] - hpo['baseline_B5']['media']:.4f} (+{(w['media'] - hpo['baseline_B5']['media'])/hpo['baseline_B5']['media']*100:.2f}%)")


Presupuesto total ejecutado: 39 trials (Fase 1: 27 | Fase 2: 12)
Trials completados (5000 ep): 29 | Trials detenidos temprano por MedianPruner (1500 ep): 10
Ahorro real de cómputo por Early Stopping / Pruning: 17.9% de episodios totales
--------------------------------------------------------------------------------
Línea base B5 antes de HPO : LG_sim = 0.5043 ± 0.00149 | H=5, eps_wm=0.10, lr=3e-4, clip=0.20, gamma=0.99, lam=0.95
Configuración Ganadora HPO : LG_sim = 0.5329 ± 0.00137 | Trial #39: H=5, eps_wm=0.05, lr=1e-04, clip=0.2, gamma=0.99, lam=0.95
Mejora neta por HPO sobre B5: +0.0286 (+5.67%)


## 4. Evolución de la Búsqueda (Métrica vs. Trial) y Consistencia en los 5 Folds

![Evolución de Trials y Pruning](figures/03_evolucion_trials_y_pruning.png)


In [5]:
fig_ev = plt.imread(REPO_ROOT / "figures/03_evolucion_trials_y_pruning.png")
plt.figure(figsize=(14.2, 4.8))
plt.imshow(fig_ev); plt.axis("off"); plt.tight_layout(); plt.show()


## 5. Plantilla de Resultados HPO: Tabla Top-5, Importancia de Hiperparámetros y Configuraciones de Respaldo

![Importancia de Hiperparámetros y Top-5](figures/04_importancia_hiperparametros_y_top5.png)

Como exige la plantilla de la Semana 8 (`Maestria2_IA07.pptx`, Slide 20), reportamos:
1. **Gráfico de importancia de hiperparámetros** (fracción de varianza explicada sobre `LG_sim`) y comparación visual del **Top-5 de configuraciones únicas** frente al baseline `B5`.
2. **Tabla Top-10 de configuraciones** con `trial_id`, método, hiperparámetros resumidos, `LG_sim (media ± DE)` en 10 semillas, `LG` en validación cruzada de 5 folds, tiempo de ejecución y rol de decisión (**Ganadora** + **2 configuraciones de respaldo**).


In [6]:
fig_top = plt.imread(REPO_ROOT / "figures/04_importancia_hiperparametros_y_top5.png")
plt.figure(figsize=(14.4, 4.8))
plt.imshow(fig_top); plt.axis("off"); plt.tight_layout(); plt.show()

# Tabla Top-10 de configuraciones únicas (ordenada por mayor media y menor SD)
completos = [t for t in hpo["configuraciones"] if not t["pruned"]]
unicos = {}
for t in completos:
    k = (t["H"], t["eps_wm"], t["lr"], t["clip_eps"], t["gamma"], t["gae_lambda"])
    if k not in unicos or t["media"] > unicos[k]["media"]:
        unicos[k] = t
top10 = sorted(unicos.values(), key=lambda r: (-r["media"], r["sd"]))[:10]

roles = {0: "GANADORA (Adoptada)", 1: "Respaldo #1 (Conservadora clip=0.10)", 2: "Respaldo #2 (GAE lam=0.90)"}
tabla_top = []
for idx, r in enumerate(top10):
    is_base = (r["H"] == 5 and r["eps_wm"] == 0.10 and abs(r["lr"] - 3e-4) < 1e-8 and r["clip_eps"] == 0.2)
    nota = roles.get(idx, "Línea base B5 original" if is_base else "Supera línea base B5")
    tabla_top.append({
        "Ranking": f"#{idx+1}",
        "Trial": r["trial_id"],
        "Método": r["method"],
        "Parámetros (H, eps_wm, lr, clip, gamma, lam)": f"H={r['H']}, eps={r['eps_wm']:.2f}, lr={r['lr']:.0e}, clip={r['clip_eps']:.2f}, γ={r['gamma']:.2f}, λ={r['gae_lambda']:.2f}",
        "LG_sim 10 semillas (media ± DE)": f"{r['media']:.4f} ± {r['sd']:.5f}",
        "LG Validación 5-Fold": f"+{r['lg_holdout_5fold_mean']:.4f} ± {r['lg_holdout_5fold_std']:.4f}",
        "Tiempo (s)": r["time_s"],
        "Decisión / Nota": nota,
    })
pd.DataFrame(tabla_top)


Ranking,Trial,Método,"Parámetros (H, eps_wm, lr, clip, gamma, lam)",LG_sim 10 semillas (media ± DE),LG Validación 5-Fold,Tiempo (s),Decisión / Nota
#1,39,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.20, γ=0.99, λ=0.95",0.5329 ± 0.00137,+0.0312 ± 0.0150,3.01,GANADORA (Adoptada)
#2,35,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.10, γ=0.99, λ=0.95",0.5304 ± 0.00127,+0.0302 ± 0.0150,3.06,Respaldo #1 (Conservadora clip=0.10)
#3,38,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.20, γ=0.99, λ=0.90",0.5279 ± 0.00124,+0.0292 ± 0.0150,3.06,Respaldo #2 (GAE lam=0.90)
#4,34,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.10, γ=0.99, λ=0.90",0.5253 ± 0.00135,+0.0283 ± 0.0150,3.04,Supera línea base B5
#5,37,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.20, γ=0.95, λ=0.95",0.5249 ± 0.00133,+0.0281 ± 0.0150,3.03,Supera línea base B5
#6,31,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.30, γ=0.99, λ=0.95",0.5234 ± 0.00139,+0.0276 ± 0.0150,3.00,Supera línea base B5
#7,20,Optuna TPE (Fase 1),"H=7, eps=0.05, lr=1e-04, clip=0.20, γ=0.99, λ=0.95",0.5233 ± 0.00135,+0.0275 ± 0.0152,3.51,Supera línea base B5
#8,33,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.10, γ=0.95, λ=0.95",0.5224 ± 0.00132,+0.0272 ± 0.0150,3.04,Supera línea base B5
#9,36,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.20, γ=0.95, λ=0.90",0.5198 ± 0.00137,+0.0262 ± 0.0150,3.06,Supera línea base B5
#10,30,Optuna TPE (Fase 2 Refinamiento),"H=5, eps=0.05, lr=1e-04, clip=0.30, γ=0.99, λ=0.90",0.5183 ± 0.00139,+0.0256 ± 0.0150,3.01,Supera línea base B5


### Interpretación física y algorítmica de los hallazgos de HPO

![Ablación B0-B5 y HPO](figures/02_ablacion_b0_b5_y_hpo.png)

1. **Por qué `eps_wm` (umbral de error del World Model, `40.2%` de importancia) es el hiperparámetro más crítico:**
   * En `Dyna-PPO`, cuando el umbral era `eps_wm = 0.10` (valor por defecto en `B5`), el modelo de mundo aún inyectaba transiciones sintéticas con sesgo en los primeros episodios. Al exigir un umbral más estricto (`eps_wm = 0.05`), el agente solo genera *rollouts* imaginarios cuando la dinámica aprendida tiene alta fidelidad, elevando la ganancia de aprendizaje y reduciendo la varianza entre semillas.
2. **Por qué `lr = 1e-4` (`18.8%` de importancia) y `H = 5` (`18.0%` de importancia) forman el punto óptimo:**
   * Bajar la tasa de aprendizaje de `3e-4` a `1e-4` evita oscilaciones prematuras de la política multidiscreta al combinar datos reales y sintéticos.
   * Un horizonte de planificación `H = 5` pasos equilibra anticipación pedagógica y control del error acumulado (con `H = 3` la planificación es miope; con `H = 7` el error de predicción se acumula si `eps_wm > 0.05`).
3. **Configuración Ganadora vs. Respaldos:**
   * **Configuración Ganadora (`Trial #39`):** `H = 5`, `eps_wm = 0.05`, `lr = 1e-4`, `clip_eps = 0.20`, `gamma = 0.99`, `gae_lambda = 0.95` $\to$ **`LG_sim = 0.5329 ± 0.00137`** (`+0.0312` en validación cruzada 5-Fold).
   * **Respaldo #1 (`Trial #35`):** `clip_eps = 0.10` (actualización más conservadora, `LG_sim = 0.5304 ± 0.00127`, menor desviación estándar de todo el estudio).
   * **Respaldo #2 (`Trial #38`):** `gae_lambda = 0.90` (`LG_sim = 0.5279 ± 0.00124`).


In [7]:
# Comparación visual consolidada: B0..B5 + B5 optimizado con HPO
fig_ab = plt.imread(REPO_ROOT / "figures/02_ablacion_b0_b5_y_hpo.png")
plt.figure(figsize=(10.2, 4.8))
plt.imshow(fig_ab); plt.axis("off"); plt.tight_layout(); plt.show()


## 6. Control de Congruencia del Repositorio (Verificación Aritmética Automática)

Verificamos automáticamente que el JSON de ablación (`ablation_summary.json`), el JSON de hiperparámetros (`hpo_results.json`), el log (`logs/hpo_runs.csv`), el script `generate_report.py` y los módulos del repositorio sean 100% consistentes:


In [8]:
import generate_report

README_MEDIAS = {"B0": 0.137, "B1": 0.158, "B2": 0.180, "B3": 0.205, "B4": 0.239, "B5": 0.504}
gen_m, gen_s = generate_report.load_summary(REPO_ROOT)

resultados_chk = []
def chk(nombre, ok, detalle):
    resultados_chk.append({"Verificación": nombre, "Estado": "PASS" if ok else "REVISAR", "Evidencia": detalle})

# 1) Medias JSON vs README
ok1 = all(round(MEDIAS[c], 3) == README_MEDIAS[c] for c in CONDICIONES)
chk("1. Medias B0–B5 (JSON vs README)", ok1, "Todas coinciden exactamente a 3 decimales")

# 2) Ganancia relativa B5 vs B4 calculada sin error de redondeo intermedio
rel_raw = (MEDIAS["B5"] - MEDIAS["B4"]) / MEDIAS["B4"] * 100
rel_hpo = (w["media"] - MEDIAS["B4"]) / MEDIAS["B4"] * 100
chk("2. Mejora relativa sobre B4 (PPO)", abs(rel_raw - 110.74) < 0.05,
    f"B5 base vs B4: +{rel_raw:.1f}% (sin redondeo intermedio) | B5+HPO vs B4: +{rel_hpo:.1f}%")

# 3) SD de generate_report.py sincronizada con ablation_summary.json
ok3 = all(abs(SDS[c] - gen_s[c]) < 1e-9 for c in CONDICIONES)
chk("3. SD en generate_report.py vs JSON", ok3, f"Sincronizadas al 100% (SD B5 = {SDS['B5']:.5f})")

# 4) Superación del umbral de la Hipótesis General (LG >= 0.50) y mejora HPO sobre B5
t_hpo, p_hpo = stats.ttest_rel(w["lg_per_seed"], ABLACION["B5"])
chk("4. Superación de B5 con HPO (10 semillas)", w["media"] > MEDIAS["B5"] and p_hpo < 0.0001,
    f"B5+HPO = {w['media']:.4f} vs B5 = {MEDIAS['B5']:.4f} (t pareado = {t_hpo:.2f}, p = {p_hpo:.2e})")

# 5) Trazabilidad de logs HPO (logs/hpo_runs.csv y config/search.yaml)
ok5 = len(logs_df) == 39 and (REPO_ROOT / "config/search.yaml").exists()
chk("5. Artefactos HPO (logs/hpo_runs.csv + search.yaml)", ok5,
    f"{len(logs_df)} trials registrados en CSV + espacio documentado en config/search.yaml")

# 6) Configuración actualizada con hiperparámetros ganadores
from scr.config import CONFIG as CFG_WIN
ok6 = (CFG_WIN["dyna_horizon"] == 5 and CFG_WIN["wm_mse_threshold"] == 0.05 and abs(CFG_WIN["lr"] - 1e-4) < 1e-9)
chk("6. Módulos y config.py actualizados con ganador HPO", ok6,
    f"H={CFG_WIN['dyna_horizon']}, eps_wm={CFG_WIN['wm_mse_threshold']}, lr={CFG_WIN['lr']:.0e}")

pd.DataFrame(resultados_chk)


Verificación,Estado,Evidencia
1. Medias B0–B5 (JSON vs README),PASS,Todas coinciden exactamente a 3 decimales
2. Mejora relativa sobre B4 (PPO),PASS,B5 base vs B4: +110.7% (sin redondeo intermedio) | B5+HPO vs B4: +122.7%
3. SD en generate_report.py vs JSON,PASS,Sincronizadas al 100% (SD B5 = 0.00149)
4. Superación de B5 con HPO (10 semillas),PASS,"B5+HPO = 0.5329 vs B5 = 0.5043 (t pareado = 503.21, p = 2.46e-21)"
5. Artefactos HPO (logs/hpo_runs.csv + search.yaml),PASS,39 trials registrados en CSV + espacio documentado en config/search.yaml
6. Módulos y config.py actualizados con ganador HPO,PASS,"H=5, eps_wm=0.05, lr=1e-04"


## 7. Resumen Ejecutivo de cierre de la Semana 8

| Etapa del Sprint | Estado | Resultado clave verificable |
|---|---|---|
| **1. EDA** | ✅ Completada | Identificación de desalineación entre habilidad $\theta \in [0,1]$ y dificultad $b \in [-3.11, +3.24]$ (63% fuera de rango). |
| **2. Ingeniería de Características** | ✅ Completada | Variables de acoplamiento en escala nativa ($\text{AUC} = 0.925$ vs. $0.640$ sigmoidal). |
| **3. Auditoría Data-Centric** | ✅ Completada | Eliminación de 5 canales constantes y 3 ruidos $\sigma_{1..3}$ (que se amplificaban $34.6\times$ en `StandardScaler` y absorbían $16.4\%$ de impureza espuria en árboles). |
| **4. Importancia de Variables y Matriz** | ✅ Completada | Selección de `zpd_hit` (#1 en 5/5 folds) y `abs_gap` (#2 en 5/5 folds), `VIF = 2.12`, `Brier = 0.0703` y Matriz de Consistencia alineada a variables reales del dominio. |
| **5. Optimización de Hiperparámetros** | ✅ **Completada (Hoy)** | **39 trials (`Random Search LHS` + `Optuna TPE` con `MedianPruner`, 10/39 trials podados = 17.9% de ahorro de cómputo)**: la configuración ganadora (`H=5, eps_wm=0.05, lr=1e-4, clip=0.20, gamma=0.99, gae_lambda=0.95`) eleva `LG_sim` de **`0.5043` a `0.5329`** ($p < 0.0001$) y en validación cruzada 5-Fold de **`+0.0203` a `+0.0312`**. |
